In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction(counter):

    return {
        "tx_id": f"TX{counter:04d}",
        "user_id": random.choice([f"u{i:02d}" for i in range(1, 6)]),
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(["Warszawa", "Krakow", "Gdansk", "Wroclaw"]),
        "category": random.choice(["elektronika", "odziez", "zywnosc", "ksiazki"]),
        "timestamp": datetime.now().isoformat()
    }

counter = 1
print("Producent uruchomiony...")
while True:
    tx = generate_transaction(counter)
    producer.send('transactions', tx)
    print(f"Wyslano: {tx['tx_id']} | {tx['user_id']} | {tx['amount']} PLN | {tx['store']}")
    counter += 1
    time.sleep(1)

Overwriting producer.py


In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction(counter):

    return {
        "tx_id": f"TX{counter:04d}",
        "user_id": random.choice([f"u{i:02d}" for i in range(1, 6)]),
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(["Warszawa", "Krakow", "Gdansk", "Wroclaw"]),
        "category": random.choice(["elektronika", "odziez", "zywnosc", "ksiazki"]),
        "timestamp": datetime.now().isoformat()
    }

counter = 1
print("Producent uruchomiony...")
while True:
    tx = generate_transaction(counter)
    producer.send('transactions', tx)
    print(f"Wyslano: {tx['tx_id']} | {tx['user_id']} | {tx['amount']} PLN | {tx['store']}")
    counter += 1
    time.sleep(1)

Overwriting producer.py


In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction(counter):

    return {
        "tx_id": f"TX{counter:04d}",
        "user_id": random.choice([f"u{i:02d}" for i in range(1, 6)]),
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(["Warszawa", "Krakow", "Gdansk", "Wroclaw"]),
        "category": random.choice(["elektronika", "odziez", "zywnosc", "ksiazki"]),
        "timestamp": datetime.now().isoformat()
    }

counter = 1
print("Producent uruchomiony...")
while True:
    tx = generate_transaction(counter)
    producer.send('transactions', tx)
    print(f"Wyslano: {tx['tx_id']} | {tx['user_id']} | {tx['amount']} PLN | {tx['store']}")
    counter += 1
    time.sleep(1)

Overwriting producer.py


In [4]:
%%file consumer_velocity.py
from kafka import KafkaConsumer
from collections import defaultdict
import json
import time

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='velocity_detector',
    auto_offset_reset='latest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_tx_times = defaultdict(list)

WINDOW_SECONDS = 60
MAX_TX_IN_WINDOW = 3

print("Nasluchuję na anomalie predkosci transakcji...")

for message in consumer:
    tx = message.value

    # TWOJ KOD
    user_id = tx["user_id"]
    tx_id   = tx["tx_id"]
    amount  = tx["amount"]
    store   = tx["store"]
    ts_iso  = tx["timestamp"]

    now = time.time()

    user_tx_times[user_id].append(now)

    user_tx_times[user_id] = [
        t for t in user_tx_times[user_id]
        if now - t <= WINDOW_SECONDS
    ]

    tx_count = len(user_tx_times[user_id])

    print(f"[INFO] {tx_id} | {user_id} | {amount:.2f} PLN | {store} | tx w oknie: {tx_count}")

    if tx_count > MAX_TX_IN_WINDOW:
        print(f"ALERT: {user_id} wykonal {tx_count} transakcji w ciagu {WINDOW_SECONDS}s! | ostatnia: {tx_id} | {amount:.2f} PLN | {store} | {ts_iso}")

Overwriting consumer_velocity.py
